# Hierarchical aDDM — sessions as random intercepts

One model **per monkey**. Sessions enter as random intercepts on all five free parameters:

| Param | Formula | Bounds | Link |
|---|---|---|---|
| `eta`   | `eta ~ 1 + (1\|session)`   | (0.0, 1.0)  | `gen_logit` |
| `kappa` | `kappa ~ 1 + (1\|session)` | (0.0, 5.0)  | `gen_logit` |
| `a`     | `a ~ 1 + (1\|session)`     | (0.1, 6.0)  | `gen_logit` |
| `b`     | `b ~ 1 + (1\|session)`     | (0.0, 3.0)  | `gen_logit` |
| `x0`    | `x0 ~ 1 + (1\|session)`    | (-2.0, 2.0) | `gen_logit` |
| `t`     | fixed `0.0` | — | — |

`b` is the **symmetric boundary-collapse slope**: boundaries sit at `±(a - b·τ)`. It was fixed at 0
in `monkey_data_recovery.ipynb`; here it is estimated, so expect `a` to land higher than the old
single-session value of 1.298 — the two trade off.

`t` stays fixed because `rt` is already non-decision-time corrected (re-referenced to first-fixation
onset, minus `MOTOR_DELAY = 0.1 s`).

**This notebook is for the cheap checks and for post-hoc analysis of saved idata.**
The long fits run through `monkey_hier_fit.py` under `monkey_hier_addm.sbatch`.

In [ ]:
# --- environment -------------------------------------------------------------
# Pin ONE GPU *before* jax is imported: a 2-GPU numpyro run can stall at 100% on
# Oscar (NCCL), and XLA otherwise pre-allocates 75% of the card up front.
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import json
import logging
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import hssm

hssm.set_floatX("float64")

SCRIPTS = Path("/users/azhan378/data/azhang/addm_hssm_paper_code/scripts")
sys.path.insert(0, str(SCRIPTS))

import monkey_hier_data as mhd
import monkey_hier_fit as mhf

# The two compat shims live in the runner so the notebook and the batch job
# cannot drift apart.
mhf.install_shims()

print("hssm", hssm.__version__, "| arviz", az.__version__)

## 1. Paths

In [ ]:
ROOT = Path(mhd.DEFAULT_ROOT)
OUT_DIR = Path(os.environ.get("SCRATCH_IDATA", f"/oscar/scratch/{os.environ['USER']}/addm_hier"))

MONKEY = "monkey_c"          # <- one monkey at a time; never both in one model
MONKEY_DIR = ROOT / MONKEY

print("monkey dir:", MONKEY_DIR, "|", len(mhd.session_files(MONKEY_DIR)), "sessions")
print("out dir   :", OUT_DIR, "(exists)" if OUT_DIR.exists() else "(not yet)")

## 2. Load — a subset first

`load_monkey` subsets by **whole sessions**, never by trials: a partial session would give the
random effect a group with an arbitrary trial count.

In [ ]:
N_SESSIONS = 3               # None -> all sessions

df = mhd.load_monkey(MONKEY_DIR, n_sessions=N_SESSIONS)
desc = mhd.describe(df)

print(json.dumps({k: v for k, v in desc.items() if k != "trials_per_session"}, indent=2))
df.head()

## 3. Build and check

`monkey_hier_fit.build_model` is the single definition of the model — the same function the batch
job calls, so what you check here is what gets fit.

The prior centres in `mhf.INTERCEPT_MU_LINK` are on the **link scale** (the linear predictor η),
not the parameter scale. Since `param = lo + (hi - lo)·sigmoid(η)` and `sigmoid` maps ℝ into (0,1),
the parameter cannot leave its bounds for any η. A negative entry just means "below the midpoint of
the bounds" — it does not imply a negative `eta`, `kappa`, `a`, or `b`.

In [ ]:
for p in mhf.PARAMS:
    lo, hi = mhf.BOUNDS[p]
    mu = mhf.INTERCEPT_MU_LINK[p]
    natural = lo + (hi - lo) / (1 + np.exp(-mu))
    print(f"{p:6} bounds={str((lo, hi)):>13}  mu_link={mu:+.2f}  -> natural centre {natural:.3f}")

In [ ]:
model = mhf.build_model(df, mhf.PARAMS)
print(model)

In [ ]:
ok, report = mhf.preflight(model, mhf.PARAMS)
print(json.dumps(report, indent=2))
assert ok, "preflight failed -- do not launch a long fit"

In [ ]:
# Cost of one gradient. This is the number that sets the wall-clock estimate;
# regressing any param forces the per-trial vmap path in the aDDM kernel.
per = mhf.time_gradient(model, report)
for draws, tune, chains in [(50, 50, 1), (200, 200, 2), (1000, 1000, 2)]:
    est = per * chains * (draws + tune) * 2 ** 6
    print(f"  draws={draws:5} tune={tune:5} chains={chains}  ->  ~{est / 3600:6.2f} h (rough)")

## 4. Launch the real fits

Two independent jobs — one per monkey. Nothing pools them.

```bash
MD=/users/azhan378/data/azhang/addm_hssm_paper_code/monkey_data/two_monkey_42_session
S=/users/azhan378/data/azhang/addm_hssm_paper_code/scripts

# dry run: build + checks + gradient timing, no sampling
sbatch --time=1:00:00 --export=ALL,MONKEY_DIR=$MD/monkey_c,DRY_RUN=1 $S/monkey_hier_addm.sbatch

# smoke: 3 sessions, tiny
sbatch --time=2:00:00 \
  --export=ALL,MONKEY_DIR=$MD/monkey_c,N_SESSIONS=3,DRAWS=50,TUNE=50,CHAINS=1 \
  $S/monkey_hier_addm.sbatch

# production: both monkeys, all sessions
for m in monkey_c monkey_k; do
  sbatch --export=ALL,MONKEY_DIR=$MD/$m $S/monkey_hier_addm.sbatch
done
```

Output lands in `$OUT_DIR` as
`addm_hier_<monkey>_<N>sess_d<draws>_t<tune>_c<chains>_seed<seed>.nc`
plus a `.json` sidecar with the arguments, session list, wall-clock, and divergence count.

## 5. Analyse a saved fit

In [ ]:
nc_files = sorted(OUT_DIR.glob("*.nc"), key=lambda p: p.stat().st_mtime, reverse=True)
for f in nc_files:
    print(f.name)
assert nc_files, f"no .nc under {OUT_DIR} yet"

NC = nc_files[0]
idata = az.from_netcdf(str(NC))
meta = json.loads(Path(str(NC)[:-3] + ".json").read_text())

print("\nloaded:", NC.name)
print("monkey:", meta["args"]["monkey"], "| sessions:", meta["data"]["n_sessions"],
      "| trials:", meta["data"]["n_trials"])
print("sampling:", meta.get("sample_seconds"), "s | divergences:", meta.get("divergences"))

In [ ]:
def posterior(idata):
    """The posterior group as a plain xarray Dataset (DataTree-safe)."""
    post = idata["posterior"]
    return post.dataset if hasattr(post, "dataset") else post


post = posterior(idata)
HIER = meta["hierarchical_params"]

pop = [f"{p}_Intercept" for p in HIER] + [
    v for v in post.data_vars if "sigma" in str(v) and "session" in str(v)
]
az.summary(idata, var_names=pop, filter_vars="like")

### The `a`–`b` ridge

A tall boundary that collapses fast mimics a low flat one. If this pair is a near-perfect ridge,
the data cannot separate boundary height from collapse rate and `b` must be reported that way.

In [ ]:
if "a" in HIER and "b" in HIER:
    az.plot_pair(idata, var_names=["a_Intercept", "b_Intercept"], kind="scatter",
                 marginals=True, scatter_kwargs={"alpha": 0.25, "s": 8})
    plt.show()
    r = np.corrcoef(post["a_Intercept"].values.ravel(), post["b_Intercept"].values.ravel())[0, 1]
    print(f"posterior corr(a_Intercept, b_Intercept) = {r:+.3f}")

### Back-transform to the natural scale

`p_s = lower + (upper - lower) * sigmoid(Intercept + u[s])` — the session-level parameter values.

In [ ]:
def session_values(post, param, bounds):
    """Per-session posterior of `param` on the natural scale."""
    lo, hi = bounds
    eta = post[f"{param}_Intercept"]
    grp = [v for v in post.data_vars if str(v).startswith(param) and "session" in str(v)
           and "sigma" not in str(v)]
    if grp:
        eta = eta + post[grp[0]]
    return lo + (hi - lo) / (1 + np.exp(-eta))


rows = []
for p in HIER:
    vals = session_values(post, p, tuple(meta["bounds"][p]))
    dim = [d for d in vals.dims if d not in ("chain", "draw")]
    if not dim:                       # pooled: one value for the whole monkey
        rows.append({"param": p, "session": "(pooled)",
                     "mean": float(vals.mean()),
                     "hdi_lo": float(az.hdi(vals.values.ravel(), hdi_prob=0.94)[0]),
                     "hdi_hi": float(az.hdi(vals.values.ravel(), hdi_prob=0.94)[1])})
        continue
    for s in vals[dim[0]].values:
        v = vals.sel({dim[0]: s}).values.ravel()
        lo_, hi_ = az.hdi(v, hdi_prob=0.94)
        rows.append({"param": p, "session": str(s), "mean": float(v.mean()),
                     "hdi_lo": float(lo_), "hdi_hi": float(hi_)})

sess_df = pd.DataFrame(rows)
sess_df.head(20)

In [ ]:
# Per-session estimates with the monkey-level mean overlaid.
params = [p for p in HIER if (sess_df.param == p).sum() > 1]
fig, axes = plt.subplots(1, len(params), figsize=(3.2 * len(params), 4.2), sharey=False)
axes = np.atleast_1d(axes)

for ax, p in zip(axes, params):
    sub = sess_df[sess_df.param == p].sort_values("mean").reset_index(drop=True)
    y = np.arange(len(sub))
    ax.errorbar(sub["mean"], y,
                xerr=[sub["mean"] - sub.hdi_lo, sub.hdi_hi - sub["mean"]],
                fmt="o", ms=4, lw=1, capsize=2)
    lo, hi = meta["bounds"][p]
    pooled = lo + (hi - lo) / (1 + np.exp(-post[f"{p}_Intercept"].values.mean()))
    ax.axvline(pooled, color="crimson", ls="--", lw=1.2, label="monkey mean")
    ax.set_yticks(y)
    ax.set_yticklabels(sub.session, fontsize=6)
    ax.set_title(p)
    ax.legend(fontsize=7)

fig.suptitle(f"{meta['args']['monkey']}: per-session posteriors (94% HDI)")
fig.tight_layout()
plt.show()

### Compare against the single-session fit

`behav_table_1.csv` — the 587-trial frame fit in `monkey_data_recovery.ipynb` — is exactly
`monkey_k/k16aug16.csv`. If this fit is monkey K, that session's back-transformed posterior should
land near the old fixed-effects estimates, shrunk toward the monkey mean.

**Caveat**: the old fit had `b = 0` fixed. Since `a` and `b` trade off, `a` should come out
*higher* here. `eta` and `kappa` are the meaningful comparison.

In [ ]:
SINGLE_SESSION = {"eta": 0.346, "kappa": 1.060, "a": 1.298, "x0": 0.028}   # b was fixed at 0

if "k16aug16" in set(sess_df.session):
    cmp = sess_df[sess_df.session == "k16aug16"].set_index("param")
    for p, old in SINGLE_SESSION.items():
        if p in cmp.index:
            r = cmp.loc[p]
            flag = "  <- a/b trade-off, expected higher" if p == "a" else ""
            print(f"{p:6} hierarchical {r['mean']:.3f} "
                  f"[{r.hdi_lo:.3f}, {r.hdi_hi:.3f}]   single-session {old:.3f}{flag}")
else:
    print("k16aug16 not in this fit (it is a monkey K session).")

### Session-effect magnitude

The group SDs are on the link scale. Translating each to the natural scale answers the actual
question: *do these parameters vary across sessions at all?*

In [ ]:
for p in HIER:
    lo, hi = meta["bounds"][p]
    sd_name = [v for v in post.data_vars
               if str(v).startswith(p) and "sigma" in str(v) and "session" in str(v)]
    if not sd_name:
        continue
    sd = float(post[sd_name[0]].mean())
    mu = float(post[f"{p}_Intercept"].mean())
    inv = lambda e: lo + (hi - lo) / (1 + np.exp(-e))
    print(f"{p:6} sigma_link={sd:.3f}  ->  natural ~[{inv(mu - sd):.3f}, {inv(mu + sd):.3f}] "
          f"around {inv(mu):.3f}")